# H&M Transaction Data: Product Recommendations 01
## Pre-process data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder

## Data loading

In [2]:
data_path = Path("../data")
customers = pd.read_csv(data_path / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(data_path / 'transactions_hm_cleaned.csv')
articles = pd.read_csv(data_path / 'articles_hm_cleaned.csv')

In [3]:
print("Cleaned Rows of Data:")
print(f"Articles: {len(articles):,}")
print(f"Customers: {len(customers):,}")
print(f"Transactions: {len(transactions):,}")

Cleaned Rows of Data:
Articles: 105,542
Customers: 1,048,575
Transactions: 1,040,101


In [4]:
# Sample data for faster execution
TRANSACTIONS_SAMPLE_SIZE = 500000
transactions_sample = transactions.sample(n=TRANSACTIONS_SAMPLE_SIZE, random_state=67)

CUSTOMER_SAMPLE_SIZE = 500000
# sample_customers = transactions_sample['customer_id'].unique()
customers_sample = customers.sample(n=CUSTOMER_SAMPLE_SIZE, random_state=67)

transactions_df = transactions_sample
customers_df = customers_sample
articles_df = articles

In [5]:
print(f"Customers: using {len(customers_df):,} out of {len(customers):,} available")
print(f"Transactions: using {len(transactions_df):,} out of {len(transactions):,} available")
print(f"Articles: using {len(articles_df):,} out of {len(articles):,} available")

Customers: using 500,000 out of 1,048,575 available
Transactions: using 500,000 out of 1,040,101 available
Articles: using 105,542 out of 105,542 available


In [85]:
PURCHASE = "Purchase"
NO_PURCHASE = "No Purchase"
LABELS = [NO_PURCHASE, PURCHASE]

## Train, Validation Data Generation

In [6]:
def build_product_recommendation_data(as_of_date, prediction_start_date, prediction_end_date):
    print(f"Data as of date: {as_of_date}")

    cfe = CustomerFeatureEngineer(customers_df=customers_df, transactions_df=transactions_df)
    customer_features = cfe.calculate_all_features(articles_df=articles_df, as_of_date=as_of_date)
    print(f"Customer rows: {len(customer_features):,}")

    pfe = ProductFeatureEngineer(articles_df=articles_df, transactions_df=transactions_df)
    product_features = pfe.calculate_all_features(as_of_date=as_of_date)
    print(f"Product rows: {len(product_features):,}")


    print(f"Prediction period: {prediction_start_date} to {prediction_end_date}")
    builder = RecommendationTrainingBuilder(transactions_df=transactions_df,
                                                customer_features_df=customer_features,
                                                product_features_df=product_features)
    data = builder.build_dataset(prediction_start=prediction_start_date,
                                            prediction_end=prediction_end_date,
                                            negative_ratio=5,
                                            random_state=67)

    print(f"Total rows: {len(data):,}")
    print(f"- Positives: {(data['purchased'] == 1).sum():,}")
    print(f"- Negatives: {(data['purchased'] == 0).sum():,}")
    return data

In [7]:
print("=========TRAINING DATA=========")
train_as_of_date = "2019-09-30"
train_prediction_start_date = "2019-10-01"
train_prediction_end_date = "2019-10-30"
train_data =  build_product_recommendation_data(train_as_of_date, train_prediction_start_date, train_prediction_end_date)

=========TRAINING DATA=========
Data as of date: 2019-09-30
Customer rows: 252,238
Product rows: 37,553
Prediction period: 2019-10-01 to 2019-10-30
Total rows: 83,100
- Positives: 13,850
- Negatives: 69,250


In [8]:
print("\n=========VALIDATION DATA=========")
val_as_of_date = "2019-10-31"
val_prediction_start = "2019-11-01"
val_prediction_end = "2019-11-30"
val_data =  build_product_recommendation_data(val_as_of_date, val_prediction_start, val_prediction_end)


=========VALIDATION DATA=========
Data as of date: 2019-10-31
Customer rows: 268,826
Product rows: 39,630
Prediction period: 2019-11-01 to 2019-11-30
Total rows: 90,432
- Positives: 15,072
- Negatives: 75,360


In [9]:
print("\n=========TEST DATA=========")
test_as_of_date = "2019-11-30"
test_prediction_start = "2019-12-01"
test_prediction_end = "2019-12-31"
test_data =  build_product_recommendation_data(test_as_of_date, test_prediction_start, test_prediction_end)


=========TEST DATA=========
Data as of date: 2019-11-30
Customer rows: 285,551
Product rows: 41,562
Prediction period: 2019-12-01 to 2019-12-31
Total rows: 85,764
- Positives: 14,294
- Negatives: 71,470


In [10]:
feature_cols = [col for col in train_data.columns if col not in ['customer_id', 'article_id', 'purchased']]

print(f"Feature Columns:")
for col in feature_cols:
    print(f"- {col}")


Feature Columns:
- sales_last_7_days
- sales_last_30_days
- days_since_first_sale
- days_since_last_sale
- avg_price
- min_price
- max_price
- product_price_std
- customer_price_std
- num_purchases
- total_spent
- days_since_last_purchase
- avg_transaction_value
- avg_days_between_purchases
- primary_department
- primary_garment_group
- category_diversity


### Categorical features
One hot encode the garment groups. Drop primary department, too many possibilities

In [11]:
print(f"Unique departments: {train_data['primary_department'].nunique()}")
print(f"Unique garment groups: {train_data['primary_garment_group'].nunique()}")

Unique departments: 207
Unique garment groups: 21


In [12]:
train_data = train_data.drop('primary_department', axis=1)
val_data = val_data.drop('primary_department', axis=1)
test_data = test_data.drop('primary_department', axis=1)

train_data = pd.get_dummies(train_data, columns=['primary_garment_group'], prefix='garment')
val_data = pd.get_dummies(val_data, columns=['primary_garment_group'], prefix='garment')
test_data = pd.get_dummies(test_data, columns=['primary_garment_group'], prefix='garment')


all_columns = set(train_data.columns).union(val_data.columns).union(test_data.columns)
print(all_columns)

train_data = train_data.reindex(columns=all_columns, fill_value=0)
val_data = val_data.reindex(columns=all_columns, fill_value=0)
test_data = test_data.reindex(columns=all_columns, fill_value=0)

{'garment_Trousers Denim', 'garment_Skirts', 'avg_transaction_value', 'garment_Special Offers', 'max_price', 'garment_Dresses Ladies', 'garment_Knitwear', 'garment_Socks and Tights', 'garment_Unknown', 'garment_Trousers', 'total_spent', 'garment_Blouses', 'garment_Shorts', 'garment_Woven/Jersey/Knitted mix Baby', 'sales_last_7_days', 'garment_Outdoor', 'purchased', 'customer_price_std', 'garment_Dressed', 'category_diversity', 'garment_Shirts', 'avg_price', 'article_id', 'days_since_last_purchase', 'customer_id', 'garment_Dresses/Skirts girls', 'garment_Jersey Basic', 'product_price_std', 'num_purchases', 'garment_Shoes', 'days_since_last_sale', 'garment_Accessories', 'min_price', 'avg_days_between_purchases', 'garment_Under-, Nightwear', 'sales_last_30_days', 'days_since_first_sale', 'garment_Swimwear', 'garment_Jersey Fancy'}


### X, y split
Outcome variable is `purchased` and 0 or 1 feature

In [187]:
X_train = train_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_train = train_data['purchased']
print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
X_val = val_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_val = val_data['purchased']
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
X_test = test_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_test = test_data['purchased']
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(83100, 36)
y_train.shape=(83100,)
X_val.shape=(90432, 36)
y_val.shape=(90432,)
X_test.shape=(85764, 36)
y_test.shape=(85764,)


## Save Processed Data
### as pickle files

In [188]:
# Save data as parquet to avoid reprocessing

processed_data_path = data_path / 'processed' / 'product_recommendation'
with open(processed_data_path / 'X_train_base.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(processed_data_path / 'y_train_base.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(processed_data_path / 'X_val_base.pkl', 'wb') as f:
    pickle.dump(X_val, f)
with open(processed_data_path / 'y_val_base.pkl', 'wb') as f:
    pickle.dump(y_val, f)
with open(processed_data_path / 'X_test_base.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(processed_data_path / 'y_test_base.pkl', 'wb') as f:
    pickle.dump(y_test, f)